In [ ]:
from sgp4.api import Satrec
from skyfield.api import EarthSatellite, load
import datetime
import pandas as pd

In [ ]:
# TO TRZEBA PODMIENIĆ ŻEBY ZAMIAST DNI BYLY GODZINY, CO 24H żeby zrobić 10dni, pierwszy pomiar ma być najlbliższe x godzin do godziny 12.00 żeby odnosić się do ground-truth z sp3, a kolejne co 24h od pierwszego

import pandas as pd
import datetime
from skyfield.api import EarthSatellite, load
from skyfield.framelib import itrs

INPUT_CSV = 'tle_C38_2025_onePerDay.csv'
OUTPUT_CSV = 'ECEF_C38_2025_long_format.csv'

SAT_ID = 'PC38' 

ts = load.timescale()
df = pd.read_csv(INPUT_CSV) 

w = []
day_offset = [0, 1, 3, 5]

for index, row in df.iterrows():
    l1 = row['TLE_LINE1'] 
    l2 = row['TLE_LINE2'] 
    tle_epoch_str = row['EPOCH']
    
    try:
        sat = EarthSatellite(l1, l2, SAT_ID, ts)
        dt_epoka = sat.epoch.utc_datetime()
        bstar = sat.model.bstar
        
        for offset in day_offset:
            # 1. Wyznaczenie daty docelowej (12:00 + offset)
            dt_target = (dt_epoka + datetime.timedelta(days=offset)).replace(hour=12, minute=0, second=0, microsecond=0)
            
            # 2. Korekta -18 sekund pod pliki SP3 (czas GPS)
            dt_target_utc = dt_target - datetime.timedelta(seconds=18)
            t_cel = ts.from_datetime(dt_target_utc)
            
            # 3. Obliczenia kinematyki
            geocentric = sat.at(t_cel)
            pos_ecef = geocentric.frame_xyz(itrs).km
            vel_ecef = geocentric.velocity.km_per_s
            
            # Tworzymy osobny wiersz dla każdego przesunięcia
            record = {
                'tle_epoch': tle_epoch_str,
                'target_epoch': dt_target.strftime('%Y-%m-%d %H:%M:%S'), # Klucz do złączenia z SP3
                'sat_id': SAT_ID,
                'days_offset': offset,
                'bstar': bstar,
                'x': round(pos_ecef[0], 6),
                'y': round(pos_ecef[1], 6),
                'z': round(pos_ecef[2], 6),
                'vx': round(vel_ecef[0], 6),
                'vy': round(vel_ecef[1], 6),
                'vz': round(vel_ecef[2], 6)
            }
            
            w.append(record)
            
    except Exception as ex:
        print(f"Błąd wiersza {index}: {ex}")

df_wyniki = pd.DataFrame(w)
df_wyniki.to_csv(OUTPUT_CSV, index=False)

print(f"Zapisano wyniki w osobrnych wierszach: {OUTPUT_CSV}")